<a href="https://colab.research.google.com/github/SUZAN1010/Formative-Assignment-for-502AZ/blob/main/502AZ_Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd  # Imports pandas for reading and inspecting tabular data.

bike = pd.read_csv("/content/train (1).csv")  # Loads the labelled bike rental CSV into a DataFrame.
print("Bike Sharing shape:", bike.shape)  # Prints the number of rows and columns.
display(bike.head())  # Displays the first five rows.

Bike Sharing shape: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [4]:
print(bike.isna().sum())  # Counts missing values in each column.

zero_wind = bike["windspeed"].eq(0)  # Marks rows with a recorded windspeed of zero.
print("Zero windspeed rows:", zero_wind.sum())  # Counts the marked rows.

zero_by_weather = zero_wind.groupby(bike["weather"]).mean() * 100  # Calculates the percentage of zero readings in each weather category.
print(zero_by_weather.round(2))  # Displays the percentages to two decimal places.

rental_total = bike["casual"] + bike["registered"]  # Adds the two types of rentals for each hour.
print("Does the sum always equal count?", rental_total.eq(bike["count"]).all())  # Checks whether the sum reveals the prediction target.

datetime      0
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
casual        0
registered    0
count         0
dtype: int64
Zero windspeed rows: 1313
weather
1    12.53
2    11.93
3     8.61
4     0.00
Name: windspeed, dtype: float64
Does the sum always equal count? True


The dataset contained no missing entries. Windspeed was recorded as zero in 1,313 rows (12.06%). The percentage varied by weather category, but this check could not determine whether the zeros represented calm conditions or missing readings. The recorded zeros were retained for the initial model comparison. casual and registered were excluded as predictors because their sum reveals the target count.

In [5]:
bike["datetime"] = pd.to_datetime(bike["datetime"])  # Converts date strings to timestamps.
bike = bike.sort_values("datetime").reset_index(drop=True)  # Orders records from earliest to latest.
bike["hour"] = bike["datetime"].dt.hour  # Extracts the hour of the day from each timestamp.

categorical = ["season", "holiday", "workingday", "weather", "hour"]  # Lists the features to one-hot encode.
numerical = ["temp", "atemp", "humidity", "windspeed"]  # Lists the numerical measurements.

X = bike[categorical + numerical]  # Selects the predictors and excludes the two rental components.
y = bike["count"]  # Selects the total rentals as the prediction target.

split = int(len(bike) * 0.8)  # Finds where the first 80% of rows ends.
X_train, X_test = X.iloc[:split], X.iloc[split:]  # Uses earlier inputs for training and later inputs for testing.
y_train, y_test = y.iloc[:split], y.iloc[split:]  # Splits rental counts at the same position.

print("Training rows:", len(X_train))  # Reports the number of training records.
print("Test rows:", len(X_test))  # Reports the number of held-out records.
print("Last training time:", bike.loc[split - 1, "datetime"])  # Shows the end of the training period.
print("First test time:", bike.loc[split, "datetime"])  # Shows the start of the test period.

Training rows: 8708
Test rows: 2178
Last training time: 2012-08-05 04:00:00
First test time: 2012-08-05 05:00:00
